# Lab: Binary Outcome Modeling with Contingency Tables
**Applied Statistics / Crop Sciences** · **75–90 minutes** · **Google Colab**

**Case study:** Weed survival following herbicide treatment (hypothetical data).

### Learning outcomes
1. Summarize binary outcomes in 2×2 and multiway contingency tables.
2. Calculate conditional probabilities, risk differences, relative risks, and odds ratios.
3. Apply chi-square and Fisher's exact tests.
4. Fit and interpret logistic regression, including treatment-by-population interaction.
5. Distinguish in-sample agreement from predictive performance.

**Instructions:** Run code cells in order; respond to the numbered questions in the blank answer cells. No external data files are required.

## Part 1 — Construct and examine a contingency table

A randomized trial records plant survival 14 days after treatment. The binary response is `Survival = 1` (survived) and `0` (died).

| Treatment | Survived | Died | Total |
|---|---:|---:|---:|
| Control | 42 | 8 | 50 |
| Treated | 18 | 32 | 50 |

**Questions:** (1) What is the experimental unit? (2) Identify response and explanatory variables. (3) What is the survival proportion in each group? (4) What would be required to interpret the difference causally?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact, chi2
import statsmodels.api as sm
import statsmodels.formula.api as smf

contingency = pd.DataFrame(
    [[42, 8], [18, 32]],
    index=['Control', 'Treated'],
    columns=['Survived', 'Died']
)
display(contingency)
display(contingency.div(contingency.sum(axis=1), axis=0).rename_axis('Treatment'))

**Your answers — Part 1**

1. 
2. 
3. 
4.

## Part 2 — Conditional probabilities and effect sizes

Define $p_C=P(Y=1\mid C)$ and $p_T=P(Y=1\mid T)$.

- **Risk difference:** $p_T-p_C$
- **Relative risk:** $p_T/p_C$
- **Odds ratio (treated/control):** $[p_T/(1-p_T)]/[p_C/(1-p_C)]$

**Questions:** Explain the sign of the risk difference; interpret relative risk and odds ratio; distinguish percentage-point from relative percentage change.

In [ ]:
p_control = contingency.loc['Control', 'Survived'] / contingency.loc['Control'].sum()
p_treated = contingency.loc['Treated', 'Survived'] / contingency.loc['Treated'].sum()
risk_difference = p_treated - p_control
relative_risk = p_treated / p_control
odds_ratio = (p_treated / (1 - p_treated)) / (p_control / (1 - p_control))
print(f'P(survival|control) = {p_control:.3f}')
print(f'P(survival|treated) = {p_treated:.3f}')
print(f'Risk difference = {risk_difference:.3f}')
print(f'Relative risk = {relative_risk:.3f}')
print(f'Odds ratio (treated/control) = {odds_ratio:.3f}')

**Your answers — Part 2**

1. 
2. 
3.

## Part 3 — Test the null hypothesis of independence

$H_0: P(Y=1\mid C)=P(Y=1\mid T)$.

We use Pearson's chi-square test without continuity correction and Fisher's exact test. **Important:** The odds ratio returned by `fisher_exact` uses the row and column order of the table; here it compares the control odds of survival with treated odds, the *reciprocal* of our earlier treated/control odds ratio.

**Questions:** Explain expected counts and degrees of freedom; compare the two p-values; distinguish statistical and biological significance.

In [ ]:
stat, p_chi, dof, expected = chi2_contingency(contingency, correction=False)
fisher_or, p_fisher = fisher_exact(contingency)
print(f'Chi-square = {stat:.4f}; df = {dof}; p = {p_chi:.3g}')
display(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))
print(f'Fisher odds ratio (control/treated) = {fisher_or:.4f}; p = {p_fisher:.3g}')
print(f'Reciprocal equals treated/control OR: {1 / fisher_or:.4f}')

**Your answers — Part 3**

1. 
2. 
3.

## Part 4 — From tables to logistic regression

$Y_i\sim\mathrm{Bernoulli}(p_i)$ and

$$\log\left(\frac{p_i}{1-p_i}\right)=\beta_0+\beta_1 T_i.$$

Here $T_i=1$ for treated, $T_i=0$ for control, and $e^{\beta_1}$ is the **treated/control odds ratio**.

**Questions:** Compare predicted probabilities with sample proportions. Verify $e^{\beta_1}$ agrees with Part 2. Interpret the sign of $\beta_1$.

In [ ]:
rows = []
for treatment, record in contingency.iterrows():
    t = int(treatment == 'Treated')
    rows.extend([{'Treatment': t, 'Survival': 1}] * int(record['Survived']))
    rows.extend([{'Treatment': t, 'Survival': 0}] * int(record['Died']))
df = pd.DataFrame(rows)
X = sm.add_constant(df[['Treatment']])
model = sm.Logit(df['Survival'], X).fit(disp=False)
print(model.summary())
print('Exponentiated coefficients:')
print(np.exp(model.params))
new_X = sm.add_constant(pd.DataFrame({'Treatment': [0, 1]}), has_constant='add')
print('Predicted P(survival), control then treated:', model.predict(new_X).round(4).tolist())

**Your answers — Part 4**

1. 
2. 
3.

## Part 5 — Three-way table: population × treatment × survival

| Population | Treatment | Survived | Died |
|---|---|---:|---:|
| A | Control | 22 | 3 |
| A | Treated | 5 | 20 |
| B | Control | 20 | 5 |
| B | Treated | 13 | 12 |

**Questions:** Calculate $P(Y=1\mid T,G)$. Which population has higher survival following treatment? What information would pooling populations obscure?

In [ ]:
counts = pd.DataFrame({
    'Population': ['A', 'A', 'B', 'B'],
    'Treatment': ['Control', 'Treated'] * 2,
    'Survived': [22, 5, 20, 13],
    'Died': [3, 20, 5, 12]
})
counts['Total'] = counts['Survived'] + counts['Died']
counts['P_survival'] = counts['Survived'] / counts['Total']
display(counts)
print('Pooling populations:')
display(counts.groupby('Treatment')[['Survived', 'Died']].sum())

**Your answers — Part 5**

1. 
2. 
3.

## Part 6 — Interaction in logistic regression

$$\mathrm{logit}(p_i)=\beta_0+\beta_1 T_i+\beta_2 G_i+\beta_3 T_iG_i.$$

Compare a main-effects model and an interaction model using a likelihood-ratio test. This tests whether the **treatment odds ratio** differs between populations, not whether the absolute treatment effect on probability differs.

**Questions:** Write R-style model notation using `X:Z`; identify the baseline group's treatment log-odds effect; interpret the interaction; evaluate whether interaction is supported.

In [ ]:
records = []
for _, r in counts.iterrows():
    for outcome, n in [(1, int(r['Survived'])), (0, int(r['Died']))]:
        records.extend([{'Population': r['Population'],
                         'Treatment': r['Treatment'],
                         'Survival': outcome}] * n)
df2 = pd.DataFrame(records)
m1 = smf.logit('Survival ~ C(Treatment) + C(Population)', data=df2).fit(disp=False)
m2 = smf.logit('Survival ~ C(Treatment) * C(Population)', data=df2).fit(disp=False)
LR = 2 * (m2.llf - m1.llf)
df_diff = int(m2.df_model - m1.df_model)
p_interaction = chi2.sf(LR, df_diff)
print(m2.summary())
print(f'Likelihood-ratio test for interaction: statistic={LR:.3f}, df={df_diff}, p={p_interaction:.4f}')

**Your answers — Part 6**

1. 
2. 
3. 
4.

## Part 7 — Visualize observed and fitted probabilities

The four-group interaction model is **saturated** (one fitted probability per group), so the model reproduces the four sample proportions exactly. This is *not* evidence of good out-of-sample prediction.

**Questions:** Why do points coincide? How would you test generalization on an independent experiment?

In [ ]:
pred = counts[['Population', 'Treatment']].copy()
pred['Observed'] = counts['P_survival']
pred['Predicted'] = m2.predict(pred)
x = np.arange(len(pred))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, pred['Observed'], s=75, label='Observed proportion')
ax.scatter(x, pred['Predicted'], s=90, marker='x', label='Model estimate')
ax.set_xticks(x, [f'{g} / {t}' for g, t in zip(pred.Population, pred.Treatment)], rotation=25)
ax.set_ylabel('Probability of survival')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

**Your answers — Part 7**

1. 
2.

## Part 8 — Independent student challenge

| Population | Treatment | Survived | Died |
|---|---|---:|---:|
| C | Control | 36 | 4 |
| C | Treated | 12 | 28 |
| D | Control | 34 | 6 |
| D | Treated | 25 | 15 |

**Submit:** (1) hypotheses, (2) contingency and conditional-probability tables, (3) risk differences, relative risks, odds ratios per population, (4) an appropriate association test, (5) logistic regression with treatment, population, and interaction, (6) observed versus fitted graph, (7) short scientific interpretation.

**Note:** Observations are treated as independent plants for this teaching example. If plants were nested within shared plots, analysis would need to account for plot as the experimental unit / within-plot dependence.

In [ ]:
# Independent challenge — begin here
challenge = pd.DataFrame({
    'Population': ['C', 'C', 'D', 'D'],
    'Treatment': ['Control', 'Treated', 'Control', 'Treated'],
    'Survived': [36, 12, 34, 25],
    'Died': [4, 28, 6, 15]
})
display(challenge)

# TODO: calculate conditional probabilities and effect sizes
# TODO: perform statistical tests and fit logistic models
# TODO: visualize and interpret the fitted probabilities

**Independent challenge — written interpretation**

**Hypotheses:**

**Statistical findings:**

**Biological conclusion and limitations:**

## Grading rubric (20 points)

| Component | Points |
|---|---:|
| Contingency tables and conditional probabilities | 4 |
| Effect sizes and interpretation | 4 |
| Hypothesis testing | 3 |
| Logistic model specification and estimation | 4 |
| Interaction interpretation | 3 |
| Scientific conclusion | 2 |